In [3]:
# 03_xgb_tuning_grid.ipynb — Cell 1
import os, json, joblib, time
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from xgboost import XGBClassifier
import sys
sys.path.append("..")
from src.transforms import get_preprocessor, select_cols_indices  # or ColumnSelector
from src.artifacts import save_joblib_with_manifest, save_json_with_manifest  # if you use artifacts helper
SEED = 42
ART_DIR = "../artifacts/feature_selection"
OUT_DIR = "../artifacts/tuning"
os.makedirs(OUT_DIR, exist_ok=True)


In [5]:
df_train = pd.read_csv("../artifacts/data/train.csv")
df_hold  = pd.read_csv("../artifacts/data/holdout.csv")
X_train = df_train.drop(columns=["num"])
y_train = df_train["num"].values
X_hold  = df_hold.drop(columns=["num"])
y_hold  = df_hold["num"].values

featmap = json.load(open(os.path.join("../artifacts/feature_selection","feature_index_map.json")))
full_feature_names = featmap["full_feature_names"]
# choose k (7 or 14 or 'all')
k = 7
topk = json.load(open(os.path.join("../artifacts/feature_selection","feature_ranks.json")))
topk = [r['feature'] for r in topk]  # ensure ordering
selected = topk[:k]
topk_indices = [full_feature_names.index(f) for f in selected]


In [7]:
selector = FunctionTransformer(func=select_cols_indices, kw_args={"indices": topk_indices}, validate=False)
pipe = Pipeline([
    ("preproc", get_preprocessor()),       # unfitted; GridSearchCV fits per fold
    ("selector", selector),
    ("xgb", XGBClassifier(use_label_encoder=False, random_state=SEED, verbosity=0))
])


In [8]:
param_grid = {
    "xgb__n_estimators": [100, 200],
    "xgb__max_depth": [3, 5],
    "xgb__learning_rate": [0.05, 0.1],
    "xgb__subsample": [0.8, 1.0],
    "xgb__colsample_bytree": [0.6, 0.8, 1.0]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grid = GridSearchCV(pipe, param_grid, scoring="roc_auc", cv=cv, n_jobs=4, verbose=2)
t0 = time.time()
grid.fit(X_train, y_train)
print("Grid done in", time.time()-t0, "s. Best:", grid.best_params_, grid.best_score_)
joblib.dump(grid.best_estimator_, os.path.join(OUT_DIR, "xgb_grid_best_k7.joblib"))
save_json_with_manifest({"best_params": grid.best_params_}, os.path.join(OUT_DIR,"xgb_grid_best_params.json"), name="xgb_grid_best_params")


Fitting 5 folds for each of 48 candidates, totalling 240 fits


ValueError: 
All the 240 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
240 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Nikhil\anaconda3\envs\heartenv\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\Nikhil\anaconda3\envs\heartenv\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Nikhil\anaconda3\envs\heartenv\Lib\site-packages\sklearn\pipeline.py", line 655, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Nikhil\anacond